# 01 - Exploration preliminaire des donnees

Ce notebook identifie automatiquement le CSV, le charge et realise des controles non destructifs. Il ne nettoie et ne modifie aucune donnee source.

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'

csv_files = sorted(
    path for path in RAW_DIR.iterdir()
    if path.is_file() and path.suffix.lower() == '.csv'
)
if not csv_files:
    raise FileNotFoundError(f'Aucun CSV trouve dans {RAW_DIR}')
if len(csv_files) > 1:
    raise RuntimeError(f'Plusieurs CSV trouves: {[p.name for p in csv_files]}')

data_path = csv_files[0]
data_path

In [ ]:
def load_csv(path):
    for encoding in ('utf-8', 'utf-8-sig', 'latin-1'):
        try:
            return pd.read_csv(path, encoding=encoding), encoding
        except UnicodeDecodeError:
            continue
    raise UnicodeError(f'Encodage non reconnu pour {path}')

df, encoding = load_csv(data_path)
print(f'Fichier : {data_path.name}')
print(f'Encodage : {encoding}')

## Apercu des 10 premieres lignes

In [ ]:
df.head(10)

## Dimensions, colonnes et types

In [ ]:
print(f'Nombre de lignes : {df.shape[0]:,}')
print(f'Nombre de colonnes : {df.shape[1]}')
print('\nColonnes :')
print(df.columns.tolist())

structure = pd.DataFrame({
    'type_pandas': df.dtypes.astype(str),
    'valeurs_uniques': df.nunique(dropna=False)
})
structure

## Valeurs manquantes

In [ ]:
missing = pd.DataFrame({
    'valeurs_manquantes': df.isna().sum(),
    'taux_manquant_pct': (df.isna().mean() * 100).round(2)
})
print(f'Total des valeurs manquantes : {int(df.isna().sum().sum())}')
missing

## Doublons

In [ ]:
exact_duplicates = int(df.duplicated().sum())
business_columns = [column for column in df.columns if column != 'Row ID']
business_duplicate_count = int(
    df.duplicated(subset=business_columns, keep='first').sum()
)
business_duplicate_mask = df.duplicated(
    subset=business_columns, keep=False
)

print(f'Doublons stricts : {exact_duplicates}')
print(f'Repetitions hors Row ID : {business_duplicate_count}')
print(f'Row ID dupliques : {int(df["Row ID"].duplicated().sum())}')
df.loc[business_duplicate_mask]

## Coherence des types

In [ ]:
mixed_types = {}
for column in df.select_dtypes(include='object').columns:
    python_types = sorted({type(value).__name__ for value in df[column].dropna()})
    if len(python_types) > 1:
        mixed_types[column] = python_types

date_checks = []
for column in ('Order Date', 'Ship Date'):
    parsed = pd.to_datetime(df[column], format='%m/%d/%Y', errors='coerce')
    date_checks.append({
        'colonne': column,
        'type_charge': str(df[column].dtype),
        'echecs_conversion': int(parsed.isna().sum()),
        'date_min': parsed.min(),
        'date_max': parsed.max()
    })

print('Colonnes avec types Python melanges :', mixed_types or 'Aucune')
print("Note : les dates restent volontairement inchangees dans df.")
pd.DataFrame(date_checks)

In [ ]:
numeric_checks = pd.Series({
    'Ship Date avant Order Date': int((
        pd.to_datetime(df['Ship Date'], format='%m/%d/%Y')
        < pd.to_datetime(df['Order Date'], format='%m/%d/%Y')
    ).sum()),
    'Sales <= 0': int((df['Sales'] <= 0).sum()),
    'Quantity <= 0': int((df['Quantity'] <= 0).sum()),
    'Discount hors [0, 1]': int(((df['Discount'] < 0) | (df['Discount'] > 1)).sum())
})
numeric_checks

## Colonnes inutiles potentielles et incoherences de format

In [ ]:
unnamed_columns = [c for c in df.columns if c.lower().startswith('unnamed')]
all_null_columns = [c for c in df.columns if df[c].isna().all()]
constant_columns = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]

whitespace_counts = {}
for column in df.select_dtypes(include='object').columns:
    values = df[column].dropna()
    count = int(values.ne(values.str.strip()).sum())
    if count:
        whitespace_counts[column] = count

print('Colonnes Unnamed :', unnamed_columns or 'Aucune')
print('Colonnes entierement vides :', all_null_columns or 'Aucune')
print('Colonnes constantes :', constant_columns or 'Aucune')
print('Espaces peripheriques :', whitespace_counts or 'Aucun')
print("\nEvaluation : Row ID est un identifiant technique; Country est constante.")
print("Postal Code devrait etre traite comme identifiant textuel lors du nettoyage.")

## Conclusion

L'exploration ne modifie aucune donnee. Les conversions de dates ci-dessus sont effectuees dans des Series temporaires uniquement. Les corrections eventuelles seront reservees a une phase de nettoyage distincte.